# Create network plots for both KarateClubNet and GEPhilNet, and their summaries.

Plots

In [ ]:
# create function that takes in a .pkl network file, plots it and saves it as a .png
import pickle
import networkx as nx
import matplotlib.pyplot as plt

def plot_network_from_pickle(pickle_file, output_image):
    # Load the network from the pickle file
    with open(pickle_file, 'rb') as f:
        G = pickle.load(f)

    # Create a new figure
    plt.figure(figsize=(20, 20))

    # Use spring layout for visualization
    pos = nx.spring_layout(G, seed=42)  # seed for reproducibility

    has_fin_freq = any('fin_freq' in attr for _, _, attr in G.edges(data=True))
    if has_fin_freq:

        nx.draw_networkx_nodes(G, pos, node_size=500, node_color='lightblue')
        nx.draw_networkx_edges(G, pos, 
                            edgelist=[(u, v) for u, v, d in G.edges(data=True) if d['fin_freq'] > 0],
                            width=[d['fin_freq'] * 0.2 for u, v, d in G.edges(data=True) if d['fin_freq'] > 0],
                            edge_color='blue', label='Financial Transactions')
    
    else:
        nx.draw_networkx_nodes(G, pos, node_size=500, node_color='lightblue')
        nx.draw_networkx_edges(G, pos, edge_color='gray')

    # no border
    plt.axis('off')

    # Save the plot as a PNG file
    plt.savefig(output_image)
    plt.close()

karate_file = '/home/nisa/Thesis/experiments/phase3/data/KarateClubNet.pkl'
GEPhil_file = '/home/nisa/Thesis/experiments/phase3/data/GePhilNet.pkl'

karate_output = '/home/nisa/Thesis/experiments/phase3/data/KarateClubNet.png'
GEPhil_output = '/home/nisa/Thesis/experiments/phase3/data/GePhilNet.png'

plot_network_from_pickle(karate_file, karate_output)
plot_network_from_pickle(GEPhil_file, GEPhil_output)

Summaries

In [ ]:
# Generate table of summaries for each network
# Number of nodes
# Number of edges
# Average degree
# Density
# Number of connected components
# Clustering coefficient
import numpy as np
from scipy.stats import skew

def degree_centralization(G):
    N    = G.number_of_nodes()
    degs = np.array([d for _,d in G.degree()])
    kmax = degs.max()
    num  = np.sum(kmax - degs)
    den  = (N-1)*(N-2)
    return num/den


def summarize_network(pickle_file):
    with open(pickle_file, 'rb') as f:
        G = pickle.load(f)
    
    if G.is_directed():
        num_of_conn_nodes = None
    else:
        num_of_conn_nodes = nx.number_connected_components(G)
    
    degrees = [d for n, d in G.degree()]

    summary = {
        'Number of nodes': G.number_of_nodes(),
        'Number of edges': G.number_of_edges(),
        'Average degree': sum(dict(G.degree()).values()) / G.number_of_nodes(),
        'Density': nx.density(G),
        'Number of connected components': num_of_conn_nodes,
        'Clustering coefficient': nx.average_clustering(G),
        "Median deg" : np.median(degrees),
        "25th/75th percentiles" : np.percentile(degrees, [25,75]),
        "Std deg" : np.std(degrees),
        "Skewness" : skew(degrees),
        "Degree centralization" : degree_centralization(G)
    }

    return summary

karate_summary = summarize_network(karate_file)
GEPhil_summary = summarize_network(GEPhil_file)
def print_summary(summary, title):
    print(f"Summary for {title}:")
    for key, value in summary.items():
        print(f"{key}: {value}")
    print("\n")

print_summary(karate_summary, "Karate Club Network")
print_summary(GEPhil_summary, "GePhil Network")

# Convergence of Optuna search - plots

In [ ]:
%pip install -U kaleido

In [ ]:
import optuna

# Load the study
storage = "sqlite:///optuna_shared_GePhil_2.db"
study_name = "AgentNet-Tuning-GePhil_2"
study = optuna.load_study(study_name=study_name, storage=storage)

# Get best trial and extract parameters
best_trial = study.best_trial
best_params = best_trial.params
print("Best Hyperparameters:", best_params)

import matplotlib.pyplot as plt
from optuna.visualization import plot_optimization_history

fig = plot_optimization_history(study)

import plotly.io as pio
pio.write_image(fig, "optuna_optimization_history_GePhil.png", format="png", width=800, height=600)

In [ ]:
import optuna

# Load the study
storage = "sqlite:///optuna_shared_KarateLink.db"
study_name = "AgentNet-Tuning-KarateLink"
study = optuna.load_study(study_name=study_name, storage=storage)

# Get best trial and extract parameters
best_trial = study.best_trial
best_params = best_trial.params
print("Best Hyperparameters:", best_params)

from optuna.visualization import plot_optimization_history

fig = plot_optimization_history(study)

import plotly.io as pio
pio.write_image(fig, "optuna_optimization_history_KarateLink.png", format="png", width=800, height=600)